#### Why AI Agents
- Booking a travel plan (need to do travel booking, hotel booking , itinerary , planning the trip)
- Ai system is intelligent system that need high level goal from uder
- Mainatain context
- Has Tools access

### characterstics
- goal driven
- autonous planning
- uses tools
- context aware
- Adpative (rethinks plan when thing changes)

In [15]:
# Model calling and intial setup
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser 
import warnings
warnings.filterwarnings("ignore") 

load_dotenv()
# Load env
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
AZURE_BASE_URL = os.getenv("AZURE_BASE_URL")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_CHAT_DEPLIOYMENT_NAME = os.getenv("AZURE_CHAT_DEPLIOYMENT_NAME")

parser = StrOutputParser()

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)

llm_openai = AzureChatOpenAI(
    model="gpt-4o-mini",                         
    deployment_name=AZURE_CHAT_DEPLIOYMENT_NAME ,  # deployment name in Azure
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_BASE_URL,
    api_version="2024-02-01"
    )
llm_openai.invoke("What are your creater, also what type of LLM are you").content
# llm_gemini.invoke("who is father of india").content

'I was created by OpenAI, and I am based on the GPT-3.5 architecture, which is a type of large language model (LLM). I am designed to understand and generate human-like text based on the input I receive. My capabilities include answering questions, providing information, and engaging in conversations across a wide range of topics. If you have any specific questions or need assistance, feel free to ask!'

In [19]:
# Making our own ai agent
from langchain_community.utilities import GoogleSerperAPIWrapper

search = GoogleSerperAPIWrapper()
search.run("Top news in india today")

"News · Hyderabad student, working at US gas station, shot dead · Shubman Gill replaces Rohit Sharma as India's ODI captain · Pak asks Trump to develop port on ... Top News · Para games in Delhi: Stray dogs bite Kenya, Japan coaches at Nehru stadium · RBI has no independence in setting inflation target, says Governor Sanjay ... Stock market today: BSE Sensex rises over 650 points; Nifty50 above 23,800 ; Trump to impose high tariffs on countries buying Venezuelan oil. Indian Dental Student, Working Part-Time At US Gas Station, Shot Dead · Hyderabad student shot dead at gas station in Texas · 28-year-old from Hyderabad shot dead ... Maserati MC Pura Coupe, Cielo With 621 HP V6 Launched In India; Check Prices · News · Kia India Appoints New Sales And Business Chiefs Amid Expansion Plans. India News · Rajasthan suspends drug controller, cracks down on key cough syrup supplier · Motorcycle and socialism in Rahul Gandhi's South America sojourn. Top News · Cyclone Shakhti Live: Rain alert for

In [24]:
# Creating a tool out of this 
from langchain_core.tools import Tool

search_tool = Tool(
        name="Intermediate_Answer",
        func=search.run,
        description="useful for when you need to ask with search",
    )


In [28]:
search_tool.invoke("Weather condition in delhi right now")

'Hourly Weather · 1 AM 82°. rain drop 0% · 2 AM 81°. rain drop 0% · 3 AM 80°. rain drop 0% · 4 AM 79°. rain drop 0% · 5 AM 79°. rain drop 0% · 6 AM 78°. rain ... Sunshine and clouds mixed. High 91F. Winds light and variable. Humidity66%. UV Index9 of 11. Sunrise6:13 am. Sunset6:06 ... New Delhi Extended Forecast with high and low temperatures ... Morning clouds. Feels Like: 98 °F. Humidity: 48%. Precipitation: Rain: 0 Snow: 0. A clear sky and light windsSunny and a gentle breezeSunny and a gentle breezeThundery showers and light windsThundery showers and light windsSunny intervals ... Block 3, Subhash Nagar, Delhi, India. As of 3:15 pm IST. 89°. Haze. Day 89° • Night 77°. Watch: See Raging Floods In Spain ... Current Weather ; RealFeel®. 95° ; Wind. NNW 5 mph ; Wind Gusts. 11 mph ; Humidity. 57% ; Indoor Humidity. 57% (Extremely Humid). Interactive Display of weather and Thunderstorm warnings. AmaranthWeather. Current Weather Across Delhi - NCR. Delhi; Gurugram; Faridabad; Gautam Budh 

In [29]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

# Fetch a langchain promt from hub
promt = hub.pull("hwchase17/react") # reAct agent

c:\Users\singh\Let's Gooooo\Langchain\.venv\Lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [30]:
# Ceate a reAct agent
agent = create_react_agent(
    llm= llm_gemini,
    tools= [search_tool],
    prompt= promt
)

In [31]:
# Creating an agent executer
agent_executer = AgentExecutor(
    agent= agent , 
    tools=[search_tool],
    verbose= True
)

In [32]:
responce = agent_executer.invoke({"input":"what is diff between weather of del and blr right now"})
print(responce)



> Entering new AgentExecutor chain...
I need to find the current weather conditions for Delhi (DEL) and Bangalore (BLR) and then compare them. I will use the intermediate tool to get the weather information for each city.
Action: Intermediate_Answer
Action Input: "current weather in Delhi"Delhi, Delhi · Current Weather. 9:18 PM. 89°F. Clear. RealFeel® 95°. Hot. RealFeel Guide. Hot. 90° to 100°. Caution advised. Possible ... Night Sky · TodayHourly14 DaysPastClimate. Currently: 89 °F. Sunny. (Weather station: New Delhi / Safdarjung, India). See more current weather. ×. Advertising ... Current Weather. 8:43 PM. 89°F. Clear. RealFeel® 94° · Looking Ahead. Thunderstorms, some strong, Sunday night. Delhi Weather Radar. Delhi Weather Radar. Static ... Hourly Weather-Rajpath Area, Central Secretariat, Delhi. As of 02:59 IST. Thursday, 2 October. 03:30. 27°. 4%. Cloudy. Feels Like32°. WindSSW 3 km/h. Today's weather forecast in Delhi is Partly cloudy. High 35°C, Low 25°C . Currently 30°C. St

In [34]:
responce = agent_executer.invoke({"input": "what is status of ind vs pak troffycontroversy"})
print(responce)



> Entering new AgentExecutor chain...
I need to find out the current status of any controversy surrounding India vs Pakistan trophy. To do this, I will use the search tool.
Action: Intermediate_Answer
Action Input: "India vs Pakistan trophy controversy"The Indian team refused to accept the trophy because it was to be presented by Pakistani Interior Minister Mohsin Naqvi. India's 2025 Asia Cup final victory over Pakistan concluded controversially as the Indian team refused the trophy from ACC Chairman Mohsin ... The captain of the winning Indian national team said it was denied the chance to lift the 2025 Asia Cup trophy after refusing to accept it ... Asian Cricket Council chairman Mohsin Naqvi continues to set absurd conditions over the Asia Cup trophy handover controversy. During the meeting, Naqvi refused to congratulate India for their win in the tournament. He was finally forced to congratulate by the BCCI ... At the presentation ceremony in Dubai on Sunday, India refused to acc

### what is reAct
- a design pattern for ai agents
- Reasoning + Acting (reAct)
- in a loop, Thought + Action + Observation

### when ReAct is useful
- multi-step problem
- Tools argumented tasks ( web search , db search etc )

In [ ]:
responce = agent_executer.invoke({"input": "what is the conversion factor bewteen inr and usd right now"})

In [39]:
responce.get("output")

'The exchange rate between INR and USD is approximately 0.011 USD per 1 INR, or approximately 88.30 to 89.57 INR per 1 USD. Please note that these rates can fluctuate throughout the day.'

### Agent vs AgentExecutor
- Agent Executor orcheastrates the whole loop
- in each loop, AgentExecutor send previos msg to agent
- tool are invoked by AgentExecutor
- AgentExecutor upadtes tool_output to thought_trace
- loops run until we get a final data